# Parent Document Retriever

This notebook is organized into categories so that every block of code is easy to follow. The underlying code is unchanged from the original notebook. What has been added is structure, grouping of related cells, and explanations of what each part does and why it matters.


## What problem does parent child retrieval solve

There is a natural tension in choosing chunk size for a retrieval system.

Small documents are useful because their embeddings can most accurately reflect their meaning. If a chunk is too long, the embedding can lose meaning, since it has to compress many different ideas into a single vector.

At the same time, chunks need to be long enough that the surrounding context of each piece of text is retained, otherwise a retrieved chunk may be too short to make sense on its own.

`ParentDocumentRetriever` strikes a balance between these two needs by splitting and storing small chunks for the purpose of searching, while keeping track of the larger document that each small chunk came from. During retrieval, it first searches over the small chunks, since their embeddings are precise, but then it looks up the parent id associated with each matching chunk and returns the larger parent document instead of the small chunk itself.

Note that the term parent document refers to whatever piece of text a small chunk originated from. Depending on how the retriever is configured, that parent can either be the entire original document, or it can itself be a medium sized chunk that was produced by an earlier splitting step.


## Category 1: Environment Setup

This category installs the packages used throughout the notebook. LangChain provides the retriever and text splitting classes, `langchain_community` provides the document loader, `sentence-transformers` provides the underlying embedding models, and `langchain_chroma` provides the Chroma vector store used to hold the child chunk embeddings.


In [ ]:
!pip install langchain

In [ ]:
!pip install -U langchain-community

In [ ]:
!pip install sentence-transformers

In [ ]:
!pip install langchain_chroma

### An optional Gemini installation

This install is left as a reference in case Gemini models are preferred over the Hugging Face and Groq models used later in the notebook. It is not required for the main flow below.


In [ ]:
####if you want to use gemini feel free to use this code.

%pip install --upgrade --quiet  google-generativeai langchain-google-genai

## Category 2: Data Ingestion

This category loads the raw text documents that will act as the source material for the parent child retrieval examples throughout the rest of the notebook.


### Loading the source documents

`TextLoader` reads a plain text file into a LangChain `Document` object. Two files are loaded here, a Paul Graham essay and a state of the union address, giving two independent full length documents to experiment with.


In [ ]:
from langchain_community.document_loaders import TextLoader

In [ ]:
loaders = [
    TextLoader(
        r"F:\Advance RAG\Data\paul_graham_essay.txt",
        encoding="utf-8"
    ),
    TextLoader(
        r"F:\Advance RAG\Data\state_of_the_union.txt",
        encoding="utf-8"
    ),
]

### Combining the loaded documents into a single list

Each loader in the list above is called in turn, and the resulting documents are collected into one flat list named `docs`, so both source files can be processed together by the retriever built later in this notebook.


In [ ]:
docs = []
for loader in loaders:
    docs.extend(loader.load())

In [ ]:
docs

## Category 3: Choosing an Embedding Model

Before building the retriever, it helps to think about which embedding model to use, since the child chunks are only useful if their embeddings capture meaning well. This category records some practical guidance on model choice before settling on the specific model actually used in the notebook.


### Practical guidance on embedding model selection

Dataset size matters. Larger datasets generally benefit from more powerful models such as MPNet based models.

Computational resources matter too. If resources are limited, a smaller model such as BGE Small En or MiniLM is often a better choice.

Task complexity is another factor. For complex tasks such as question answering or text summarization, MPNet based models are often preferred.

Embedding dimensionality also differs between models, since different models produce embeddings of varying dimensions, and the right choice depends on the requirements of whatever downstream task will use those embeddings.

There is also a performance versus efficiency trade off to consider, depending on whether the priority is high accuracy or faster processing.

Experimentation is key. Trying a few different models and evaluating their performance on the specific task and dataset at hand is the most reliable way to find the best fit.

A few reference terms and links are useful here. MTEB stands for Massive Text Embedding Benchmark. MPNet stands for Masked and Permuted Pre training for Language Understanding. BGE stands for BAAI general embedding, from BAAI, whose page can be found at huggingface.co/BAAI. General sentence transformer models can be found at huggingface.co/sentence-transformers, the MTEB leaderboard is at huggingface.co/spaces/mteb/leaderboard, and a blog post explaining MTEB is available at huggingface.co/blog/mteb.

As a rule of thumb, the all mpnet base v2 model provides the best quality, while all MiniLM L6 v2 is about five times faster and still offers good quality, which makes it a practical default for experimentation.


### A commented out alternative using an MPNet based model

This cell shows how a larger, higher quality embedding model could be loaded instead, running on a GPU device. It is left commented out and unused, since the notebook proceeds with the faster MiniLM model below.


In [ ]:
'''# specify embedding model (using huggingface sentence transformer)
from langchain.embeddings import HuggingFaceEmbeddings
embedding_model_name = "sentence-transformers/all-mpnet-base-v2"
model_kwargs = {"device": "cuda"}
embeddings = HuggingFaceEmbeddings(
  model_name=embedding_model_name,
  model_kwargs=model_kwargs
)'''

### Loading the embedding model actually used in this notebook

`HuggingFaceEmbeddings` wraps the `sentence-transformers/all-MiniLM-L6-v2` model, the fast and lightweight model mentioned above, which is used to embed every child chunk throughout the rest of the notebook. Calling `embed_query` on a simple test string confirms that the embedding model loads correctly and returns a numeric vector.


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
embeddings.embed_query("Hello world")

## Category 4: A Basic Parent Document Retriever Using Whole Documents as Parents

This category builds the simplest version of a parent document retriever, where the parent documents are the entire original files loaded in Category 2, and only a child splitter is used to break those files into small searchable chunks.


### Creating the child text splitter

`RecursiveCharacterTextSplitter` with a chunk size of 400 characters is used to create the child documents, the small chunks that will actually be embedded and searched. Because no parent splitter is supplied in this section, the whole original document from Category 2 will act as the parent for every child chunk derived from it.


In [ ]:
# This text splitter is used to create the child documents
from langchain_text_splitters import RecursiveCharacterTextSplitter
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400)

### Setting up storage for the vector index and the parent documents

`Chroma` is the vector store that will hold the embedded child chunks. `InMemoryStore` is a simple key value store that will hold the full parent documents, keyed by an id that each child chunk's metadata points back to.


In [ ]:
from langchain.storage import InMemoryStore
from langchain_chroma import Chroma

In [ ]:
vectorstore = Chroma(
    collection_name="full_documents", embedding_function=embeddings
)

In [ ]:
store = InMemoryStore()

### Assembling the retriever

`ParentDocumentRetriever` is given the vector store to search over, the docstore to look up parents in, and the child splitter to use when breaking documents into chunks. Since no `parent_splitter` argument is passed here, each original document is stored as is in `store`, while its child chunks are embedded and stored in `vectorstore`.


In [ ]:
from langchain.retrievers import ParentDocumentRetriever
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

### Adding the documents to the retriever

Calling `add_documents` runs each of the two source documents through the child splitter, embeds every resulting chunk into `vectorstore`, and stores the original full documents in `store` under automatically generated ids, since `ids` is passed as `None`. Listing the keys in the store afterward confirms how many parent documents were stored.


In [ ]:
retriever.add_documents(docs, ids=None)

In [ ]:
list(store.yield_keys())

### Testing this basic retriever

The same question about a Supreme Court nomination used throughout the earlier notebooks in this series is used here. Even though the retriever searched over small four hundred character child chunks, the result it returns is the full original document, since no parent splitter was used and the whole document is the registered parent. Printing the content length shows just how large that returned parent document actually is compared to a single small chunk.


In [ ]:
retrieved_docs= retriever.invoke("What did the president say about Ketanji Brown Jackson")

In [ ]:
print(retrieved_docs[0].page_content)

In [ ]:
print(len(retrieved_docs[0].page_content))

## Category 5: Using Both a Parent Splitter and a Child Splitter

Returning an entire original document as the parent, as in Category 4, can be excessive when the source documents are long. This category refines the approach by introducing a second splitter that breaks each original document into medium sized parent chunks first, and only then splits those parent chunks further into small child chunks. This way, a retrieved parent is a reasonably sized chunk of the original document rather than the entire file.


### Creating the child and parent splitters

The child splitter here uses a chunk size of 500 characters, so it should produce chunks smaller than whatever parent chunk they came from. The parent splitter uses a much larger chunk size of 2000 characters, so each parent chunk is a sizeable but still bounded piece of the original document, rather than the document in its entirety.


In [ ]:
# It should create documents smaller than the parent
child_splitter = RecursiveCharacterTextSplitter(chunk_size=500)

In [ ]:
# This text splitter is used to create the parent documents
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000)

### Preparing a fresh vector store and docstore

A new Chroma collection and a new in memory store are created for this second retriever, so the child chunks and parent chunks from this section do not mix with the whole document parents from Category 4. The call to `delete_collection` clears out any existing data under the same collection name before the fresh collection is created, which keeps repeated runs of this notebook from accumulating duplicate entries.


In [ ]:
vectorstore1.delete_collection()

In [ ]:
vectorstore1 = Chroma(
    collection_name="full_documents",
    embedding_function=embeddings
)

In [ ]:
# The storage layer for the parent documents
store1 = InMemoryStore()

### Assembling the retriever with both splitters

This second `ParentDocumentRetriever`, named `retriever2`, is configured with both a `child_splitter` and a `parent_splitter`. Now, each original document is first broken into 2000 character parent chunks, and each of those parent chunks is further broken into 500 character child chunks for embedding. This gives more granular, better bounded parent documents compared to Category 4.


In [ ]:
retriever2 = ParentDocumentRetriever(
    vectorstore=vectorstore1,
    docstore=store1,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

In [ ]:
retriever2.add_documents(docs)

### Comparing how many parent documents each approach produced

Counting the keys in each docstore shows the practical effect of adding a parent splitter. `store1`, which used a 2000 character parent splitter, ends up with many more parent entries than `store`, which stored each original file as a single parent, since the same source text is now broken into several medium sized parent chunks rather than kept whole.


In [ ]:
len(list(store1.yield_keys()))

In [ ]:
len(list(store.yield_keys()))

## Category 6: Directly Exploring Child and Parent Documents

Rather than only calling the retriever and looking at its final answer, this category reaches into the vector store and the docstore directly, to make the relationship between a small child chunk and its larger parent chunk completely visible.


### Viewing all child documents

`vectorstore1.get()` returns every child chunk stored in the vector store along with its metadata, giving a full inventory of the small chunks that this second retriever actually searches over. The total count and the content of the very first child chunk are printed to get a concrete sense of what these chunks look like.


In [ ]:
# Get all child documents stored in the vector store
children = vectorstore1.get()

# Total number of child documents
print("Total Child Documents:", len(children["documents"]))

In [ ]:
# Display the first child document (small chunk)
print(children["documents"][0])

### Inspecting a child document's metadata

Each child chunk's metadata contains a `doc_id` field, which is exactly the key needed to look up that chunk's parent in the docstore. Printing the metadata of the first child confirms this id is present and shows what it looks like.


In [ ]:
# Display metadata of the first child document
# Metadata contains the parent document ID (doc_id)
print(children["metadatas"][0])

### Viewing all parent documents

`store1.yield_keys()` lists every parent id known to the docstore. Printing the total count and the first five ids gives a sense of scale and shows what these ids look like, ahead of using one of them to fetch an actual parent document in the next step.


In [ ]:
# Get all parent document IDs stored in the docstore
parent_ids = list(store1.yield_keys())

print("Total Parent Documents:", len(parent_ids))
print(parent_ids[:5])   # Show first 5 IDs

### Following one child chunk to its parent

The `doc_id` recorded in the first child chunk's metadata is used to look up that exact parent document in `store1` using `mget`. Printing the parent's content shows the larger, more context rich piece of text that this small child chunk actually belongs to.


In [ ]:
# Extract the parent ID from the child metadata
parent_id = children["metadatas"][0]["doc_id"]

# Retrieve the corresponding parent document
parent_doc = store1.mget([parent_id])[0]

print(parent_doc.page_content)

### Placing the child and its parent side by side

Printing the same child chunk and its corresponding parent document together, with clear section headers, makes the size difference and the added context in the parent immediately visible in a single glance.


In [ ]:
# Display the child document
print("========== CHILD DOCUMENT ==========\n")
print(children["documents"][0])

# Display the corresponding parent document
parent_id = children["metadatas"][0]["doc_id"]
parent_doc = store1.mget([parent_id])[0]

print("\n========== PARENT DOCUMENT ==========\n")
print(parent_doc.page_content)

### Looking at several child parent pairs at once

Looping over the first five child chunks and printing each one alongside the first five hundred characters of its parent gives a broader view of how consistently this child to parent relationship holds across different parts of the dataset, rather than relying on a single example.


In [ ]:
# Display the first 5 child-parent pairs
for i in range(5):

    child = children["documents"][i]
    parent_id = children["metadatas"][i]["doc_id"]
    parent = store1.mget([parent_id])[0]

    print(f"\n========== Pair {i+1} ==========")
    print("\nChild:\n")
    print(child)

    print("\nParent:\n")
    print(parent.page_content[:500])  # Show first 500 characters

### A diagram of how the pieces fit together

This diagram summarizes the flow explored in this category. An original document is first broken up by the parent splitter into parent documents, which are stored in `store1`. Each parent document is then broken up further by the child splitter into child documents, which are embedded and stored in `vectorstore1`. Every child document keeps a record of its parent's id, so that a similarity search over the child documents can always be traced back to the correct parent document, which is what ultimately gets returned to a downstream language model.

```text
Original Document
        │
        ▼
Parent Splitter
        │
        ▼
Parent Documents (Stored in `store1`)
        │
        ▼
Child Splitter
        │
        ▼
Child Documents (Stored in `vectorstore1`)
        │
        ▼
Each child stores its parent ID (`doc_id`)
        │
        ▼
Similarity Search
        │
        ▼
Child Retrieved
        │
        ▼
Parent Document Returned to the LLM
```


## Category 7: Comparing Child Chunk Retrieval Against Parent Document Retrieval

This category runs the same question through both the raw vector store, which returns small child chunks, and the full `ParentDocumentRetriever`, which returns the larger parent documents, so the practical difference between the two can be seen directly.


### Retrieving through the parent document retriever

Invoking `retriever2` with the Supreme Court nomination question returns parent chunks rather than the raw small chunks, and printing the length of the first result's content confirms that it is noticeably larger than an individual four hundred or five hundred character child chunk.


In [ ]:
retrieved_docs2= retriever2.invoke("What did the president say about Ketanji Brown Jackson")

In [ ]:
retrieved_docs2

In [ ]:
len(retrieved_docs2[0].page_content)

### Retrieving the raw child chunks directly from the vector store

Calling `similarity_search` directly on `vectorstore1`, bypassing the parent document retriever entirely, returns the small child chunks themselves along with their metadata. Comparing these chunks against the parent documents returned above makes the size and context difference concrete.


In [ ]:
# Retrieve the child chunks from the vector store
child_docs = vectorstore1.similarity_search(
    "What did the president say about Ketanji Brown Jackson",
    k=4
)

# Display the retrieved child chunks
for i, doc in enumerate(child_docs, 1):
    print(f"\n========== Child Chunk {i} ==========\n")
    print(doc.page_content)
    print("\nMetadata:", doc.metadata)

### Placing a raw child chunk next to its parent document

This final comparison retrieves both the small child chunks and the parent documents for the exact same question, then prints the top child chunk directly above the top parent document, so the contrast between what similarity search actually matched on and what is ultimately handed back is as clear as possible.


In [ ]:
# Retrieve child chunks
child_docs = vectorstore1.similarity_search(
    "What did the president say about Ketanji Brown Jackson",
    k=3
)

# Retrieve parent documents
parent_docs = retriever2.invoke(
    "What did the president say about Ketanji Brown Jackson"
)

print("========== CHILD CHUNK ==========\n")
print(child_docs[0].page_content)

print("\n" + "=" * 80 + "\n")

print("========== PARENT DOCUMENT ==========\n")
print(parent_docs[0].page_content)

### A diagram of the retrieval flow

This diagram summarizes what happens whenever `retriever2.invoke` is called with a question. The query is embedded and compared against the child chunks stored in the vector store. Once the closest child chunks are found, `ParentDocumentRetriever` maps each child chunk's `doc_id` back to its parent document, and it is those parent documents, not the raw child chunks, that are ultimately returned to the language model.

```text
Query
   │
   ▼
Vector Store
   │
   ▼
Child Chunks Retrieved
   │
   ▼
ParentDocumentRetriever
   │
   ▼
Maps child `doc_id` → Parent Document
   │
   ▼
Returns Parent Documents to the LLM
```

To summarize the two retrieval methods used above in one line each: calling `vectorstore1.similarity_search(...)` returns child chunks, while calling `retriever2.invoke(...)` returns parent documents.


## Category 8: Generating a Final Answer With a Language Model

This final category connects the parent document retriever built in Category 5 to an actual language model, completing the pipeline from raw documents all the way to a natural language answer grounded in the retrieved parent documents.


### Loading the Groq API key

The Groq API key is stored in a local `.env` file and loaded with `python-dotenv`, so the language model call below does not require any credentials to be hard coded in the notebook.


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

### Creating the chat model and confirming it works

`ChatGroq` is configured to use the `llama-3.3-70b-versatile` model with temperature set to zero, so its behavior stays as consistent and factual as possible. A quick standalone call, asking the model to write a ballad about LangChain, confirms the connection and credentials are working correctly before the model is wired into any retrieval chain.


In [ ]:
from langchain_groq import ChatGroq
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=GROQ_API_KEY,
    temperature=0,
)

result = llm.invoke("Write a ballad about LangChain")
print(result.content)

### Building the retrieval question answering chain

`RetrievalQA.from_chain_type` connects the language model with `retriever2`, the parent document retriever from Category 5, using the stuff chain type, which simply inserts all retrieved parent documents directly into the prompt. The same Supreme Court nomination question used throughout this notebook is defined here as the query to ask.


In [ ]:
from langchain.chains import RetrievalQA
from langchain.llms import OpenAI

qa = RetrievalQA.from_chain_type(llm=llm,
                                 chain_type="stuff",
                                 retriever=retriever2)

query = "What did the president say about Ketanji Brown Jackson"


### Running the chain to get a final answer

Calling `qa.run` performs parent document retrieval and then asks the language model to answer the question using the retrieved parent documents as context, returning a complete natural language answer grounded in the state of the union address.


In [ ]:
qa.run(query)

### Reading the answer

The president said that he nominated Circuit Court of Appeals Judge Ketanji Brown Jackson to serve on the United States Supreme Court, four days prior to the speech. He described her as one of the nation's top legal minds, someone who would continue Justice Breyer's legacy of excellence. He also mentioned that she has broad support, including from the Fraternal Order of Police and from former judges appointed by both Democrats and Republicans. He further highlighted her background as a former top litigator in private practice, a former federal public defender, and someone who comes from a family of public school educators and police officers, describing her as a consensus builder.


## Conclusion

This notebook explored the parent document retrieval pattern as a practical answer to the tension between chunk size and preserved context in retrieval systems.

The first approach, built in Category 4, used only a child splitter and treated each entire original document as its own parent. This guaranteed that no context was ever lost, since the full document was always available, but it also meant that a single matched chunk could pull in an entire, potentially very long document as context for the language model.

The second approach, built in Category 5, introduced a parent splitter as well, so that each original document was first broken into medium sized parent chunks, and only those parent chunks were further broken into small child chunks for embedding. This produced parent documents that were meaningfully smaller and more focused than a whole file, while still being noticeably larger and more context rich than the small child chunks that similarity search actually matched against.

Exploring the vector store and the docstore directly in Category 6 made this relationship concrete, showing exactly how each small child chunk carries a `doc_id` in its metadata that maps back to a specific parent chunk, and Category 7 confirmed the practical effect by comparing raw child chunk retrieval against full parent document retrieval on the same question. Finally, Category 8 connected the parent document retriever to a Groq hosted language model through a `RetrievalQA` chain, producing a complete, accurately grounded answer about a real passage in the state of the union address.

The overall takeaway is that parent document retrieval lets a system search with the precision of small embeddings while still answering with the context of larger passages, and choosing whether the parent should be a whole document or a bounded parent chunk is itself a useful tuning decision depending on how long the source documents are and how much surrounding context a typical question actually needs.
